In [ ]:
import matplotlib.pyplot as plt
from algbench import read_as_pandas
from shapely import MultiPolygon, Polygon, unary_union
from shapely.plotting import plot_polygon
from tspn_bnb2.schemas import AnnotatedInstance

In [ ]:
def read_row(row):
    if row["result"]["output_instance"] is None:
        return None
    return {
        "output_instance": AnnotatedInstance.model_validate_json(row["result"]["output_instance"]),
        "annotated_instance": AnnotatedInstance.model_validate_json(
            row["result"]["annotated_instance"]
        ),
        "instance_name": row["parameters"]["args"]["kwargs"]["instance_name"],
        "instance": AnnotatedInstance.model_validate_json(
            row["parameters"]["args"]["kwargs"]["instance_json"]
        ),
        "simplification_time": row["result"]["simplification_time"],
        "remove_holes": row["parameters"]["args"]["alg_params"]["remove_holes"],
        "convex_hull_fill": row["parameters"]["args"]["alg_params"]["convex_hull_fill"],
        "remove_supersites": row["parameters"]["args"]["alg_params"]["remove_supersites"],
    }


benchmark = read_as_pandas("results_simplification", read_row)
print("loaded", len(benchmark), "simplifications")

In [ ]:
n_shown = 10
for _, row in benchmark[(benchmark["remove_supersites"])].iterrows():
    if all(len(poly.interiors) == 0 for poly in row["annotated_instance"].polygons):
        continue

    n_shown -= 1
    if n_shown == 0:
        break

    fig, axs = plt.subplots(ncols=3, figsize=(6, 3))

    for ax in axs.flat:
        ax.set_axis_off()
        ax.set_aspect("equal")

    print(
        row["instance_name"], row["convex_hull_fill"], row["remove_holes"], row["remove_supersites"]
    )

    for polygon in row["instance"].polygons:
        # assert polygon.is_valid
        plot_polygon(polygon=polygon, ax=axs[0], add_points=False, color="grey")
        plot_polygon(polygon=polygon, ax=axs[1], add_points=False, color="grey")

    for polygon in row["output_instance"].polygons:
        # assert polygon.is_valid, f"{polygon.wkt}"
        plot_polygon(polygon=polygon, ax=axs[2], add_points=False, color="grey")

    input_polys = unary_union(row["instance"].polygons)
    output_polys = unary_union(row["output_instance"].polygons)

    removed_parts = input_polys.difference(output_polys)
    added_parts = output_polys.difference(input_polys)

    if isinstance(removed_parts, Polygon):
        removed_parts = [removed_parts]
    elif isinstance(removed_parts, MultiPolygon):
        removed_parts = removed_parts.geoms

    if isinstance(added_parts, Polygon):
        added_parts = [added_parts]
    elif isinstance(added_parts, MultiPolygon):
        added_parts = added_parts.geoms

    for polygon in removed_parts:
        if not isinstance(polygon, Polygon):
            continue
        plot_polygon(polygon=polygon, ax=axs[1], add_points=False, color="red", alpha=0.3)

    for polygon in added_parts:
        if not isinstance(polygon, Polygon):
            continue
        plot_polygon(polygon=polygon, ax=axs[1], add_points=False, color="green", alpha=0.3)

    fig.show()